## LG Aimers 9기 — Trackman 익명 선수 연결 가능성 감사 (Colab)

메인 데이터의 익명 `pitcher_id`와 Trackman의 `pitcher_trackman_id`를 바로 합치지 않는다. 정답을 전혀 읽지 않고 공식 `asof_pitcher_*` 구종 이력, 활동 시즌, 투구 수, 손잡이와 팀 일관성만으로 재현 가능한 연결 후보를 찾는다. 전체·초기 시즌·최근 시즌·시즌별 단독 매칭이 모두 안정적인 경우만 고신뢰 후보로 인정한다.

### 0. Drive 마운트와 입력 파일 확인

In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

candidates = [
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path.cwd(),
]
PROJECT_ROOT = next((path for path in candidates if (path / "data/train.csv").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("LG_aimers_9th 프로젝트 또는 data/train.csv를 찾지 못했습니다.")

TRAIN_PATH = PROJECT_ROOT / "data/train.csv"
TRACKMAN_PATH = PROJECT_ROOT / "data/trackman_history.csv"
SCRIPT_PATH = PROJECT_ROOT / "experiments/06_trackman_linkage_audit.py"
OUTPUT_ROOT = PROJECT_ROOT / "outputs/trackman_linkage_audit_v1"
for path in (TRAIN_PATH, TRACKMAN_PATH, SCRIPT_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("train.csv (GB):", round(TRAIN_PATH.stat().st_size / 1e9, 3))
print("trackman_history.csv (GB):", round(TRACKMAN_PATH.stat().st_size / 1e9, 3))
print("CPU threads:", os.cpu_count())
print("Python:", sys.version.split()[0])

### 1. 감사 원칙과 사전 고정 기준

`control_success`는 읽지 않는다. 메인 데이터의 누적 구종 비율에서 직전 투구군을 복원하고 Trackman의 시즌별 구종 분포와 비교한다. 먼저 매우 엄격한 구종 지문으로 익명 팀 대응을 만들고, 그 팀 대응을 2차 일관성 조건으로 추가한다. 기준은 모델 점수를 보기 전에 고정한다.

In [ ]:
import json
import subprocess
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SETTINGS = {
    "min_pitches_per_season": 30,
    "min_overlap_seasons": 2,
    "max_pitchmix_rmse": 0.055,
    "min_score_margin": 0.015,
    "min_activity_jaccard": 0.50,
    "min_activity_recall": 0.75,
    "min_season_vote_share": 0.60,
    "min_team_match_rate": 0.80,
    "proceed_min_row_coverage": 0.60,
    "partial_min_row_coverage": 0.30,
}
pd.Series(SETTINGS, name="value").to_frame()

판정은 다음 세 단계로 보수적으로 나눈다.

- `proceed_high_confidence_only`: 고신뢰 선수만 Trackman 피처 CV로 진행
- `partial_mapping_experiment_only`: 제출 사용 금지, fallback을 포함한 OOF 진단만 허용
- `reject_player_level_mapping`: 선수별 연결을 포기하고 저해상도 집계 또는 F 구조 변화 실험으로 이동

### 2. Trackman 연결 감사 실행

In [ ]:
command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--train-path", str(TRAIN_PATH),
    "--trackman-path", str(TRACKMAN_PATH),
    "--output-dir", str(OUTPUT_ROOT),
    "--min-pitches-per-season", str(SETTINGS["min_pitches_per_season"]),
    "--min-overlap-seasons", str(SETTINGS["min_overlap_seasons"]),
    "--max-pitchmix-rmse", str(SETTINGS["max_pitchmix_rmse"]),
    "--min-score-margin", str(SETTINGS["min_score_margin"]),
    "--min-activity-jaccard", str(SETTINGS["min_activity_jaccard"]),
    "--min-activity-recall", str(SETTINGS["min_activity_recall"]),
    "--min-season-vote-share", str(SETTINGS["min_season_vote_share"]),
    "--min-team-match-rate", str(SETTINGS["min_team_match_rate"]),
    "--proceed-min-row-coverage", str(SETTINGS["proceed_min_row_coverage"]),
    "--partial-min-row-coverage", str(SETTINGS["partial_min_row_coverage"]),
]
result = subprocess.run(command, check=True, text=True)
result

### 3. 실행 결과와 구종 이력 복원 감사

In [ ]:
with (OUTPUT_ROOT / "latest_run.json").open(encoding="utf-8") as handle:
    latest = json.load(handle)
RUN_DIR = Path(latest["run_dir"])
summary = latest["summary"]
decision = summary["decision"]

decision_table = pd.DataFrame([
    {
        "decision": decision["decision"],
        "high_confidence_pitchers": decision["high_confidence_pitchers"],
        "pitcher_coverage": decision["pitcher_coverage"],
        "row_coverage": decision["row_coverage"],
        "minimum_season_coverage": decision["minimum_season_row_coverage"],
        "median_pitchmix_rmse": decision["median_pitchmix_rmse"],
        "median_vote_share": decision["median_season_vote_share"],
        "median_team_match": decision["median_team_match_rate"],
        "duplicate_trackman_ids": decision["duplicate_trackman_ids"],
    }
])
print("RUN_DIR:", RUN_DIR)
decision_table.T

In [ ]:
with (RUN_DIR / "reconstruction_audit.json").open(encoding="utf-8") as handle:
    reconstruction = json.load(handle)
main_audit = reconstruction["main"]
track_audit = reconstruction["trackman"]
audit_table = pd.DataFrame([
    {
        "dataset": "main",
        "rows": main_audit["rows"],
        "pitchers": main_audit["pitchers"],
        "target_read": main_audit["target_column_read"],
        "recoverable_fraction": main_audit["recovered_fraction_of_consecutive"],
        "extra": f"n increment=1: {main_audit['n_increment_one_fraction']:.6f}",
    },
    {
        "dataset": "trackman",
        "rows": track_audit["rows"],
        "pitchers": track_audit["pitchers"],
        "target_read": False,
        "recoverable_fraction": np.nan,
        "extra": f"core group fraction: {track_audit['core_fraction']:.6f}",
    },
])
audit_table

복원률이 높아도 선수 ID 연결이 자동으로 증명되는 것은 아니다. 복원은 메인 데이터 내부 누적값의 일관성 검사이고, 연결 신뢰도는 아래의 상호 최근접·시즌 재현·팀 일관성으로 별도 판정한다.

### 4. 고신뢰 매핑 범위와 임계값 민감도

In [ ]:
mapping = pd.read_csv(RUN_DIR / "pitcher_mapping_audit.csv", dtype={"pitcher_id": "string", "pitcher_trackman_id": "string"})
coverage = pd.read_csv(RUN_DIR / "coverage_by_season.csv")
sensitivity = pd.read_csv(RUN_DIR / "threshold_sensitivity.csv")
team_mapping = pd.read_csv(RUN_DIR / "team_season_mapping.csv")

display(coverage)
display(sensitivity.sort_values(["max_pitchmix_rmse", "min_score_margin"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(coverage["season"].astype(str), coverage["row_coverage"], color="#4472C4")
axes[0].axhline(SETTINGS["proceed_min_row_coverage"], color="#C00000", linestyle="--", label="proceed threshold")
axes[0].set_ylim(0, 1)
axes[0].set_title("High-confidence row coverage by season")
axes[0].legend()

for margin, group in sensitivity.groupby("min_score_margin"):
    axes[1].plot(group["max_pitchmix_rmse"], group["row_coverage"], marker="o", label=f"margin≥{margin:.3f}")
axes[1].axhline(SETTINGS["partial_min_row_coverage"], color="#ED7D31", linestyle="--", label="partial threshold")
axes[1].axhline(SETTINGS["proceed_min_row_coverage"], color="#C00000", linestyle="--", label="proceed threshold")
axes[1].set_ylim(0, 1)
axes[1].set_title("Coverage under predeclared threshold grid")
axes[1].set_xlabel("Maximum pitchmix RMSE")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### 5. 고신뢰 후보와 탈락 원인

In [ ]:
check_columns = [
    "pitcher_id", "pitcher_trackman_id", "main_rows",
    "pitchmix_rmse", "score_margin", "overlap_seasons",
    "mutual_top1", "early_late_stable", "season_vote_share",
    "team_match_rate", "high_confidence",
]
display(mapping.loc[mapping["high_confidence"], check_columns].head(30))

failure_summary = pd.Series({
    "all_main_pitchers": len(mapping),
    "mutual_top1": int(mapping["mutual_top1"].fillna(False).sum()),
    "early_late_stable": int(mapping["early_late_stable"].fillna(False).sum()),
    "fingerprint_high_confidence": int(mapping["fingerprint_high_confidence"].fillna(False).sum()),
    "final_high_confidence": int(mapping["high_confidence"].fillna(False).sum()),
}, name="pitchers").to_frame()
failure_summary

### 6. 익명 팀 대응의 내부 일관성

팀 대응은 외부 선수 명단으로 만든 것이 아니다. 엄격한 1차 구종 지문에서 나온 선수들만 투표에 사용하며, 시즌별 지지 선수 수와 purity를 함께 남긴다.

In [ ]:
team_columns = [
    "main_team_id", "season", "inferred_track_team_family",
    "support_pairs", "purity", "reliable_team_mapping",
]
team_mapping[team_columns].sort_values(["main_team_id", "season"])

### 7. 자동 판정과 다음 실험

In [ ]:
print("Decision:", decision["decision"])
print("Reasoned next step:", decision["next_step"])
print("Warning:", decision["identity_warning"])

if decision["decision"] == "proceed_high_confidence_only":
    print("다음: 06b에서 고신뢰 매핑 행만 Trackman 피처를 부여하고 나머지는 결측/fallback으로 처리한다.")
elif decision["decision"] == "partial_mapping_experiment_only":
    print("다음: 제출에는 쓰지 않고 mapped-only + fallback OOF 진단을 한 번만 수행한다.")
else:
    print("다음: 선수별 Trackman 연결은 중단하고 07 F 구조 변화 실험으로 이동한다.")

### 8. 검토용 결과 ZIP 생성

In [ ]:
review_files = [
    "summary.json",
    "configuration.json",
    "reconstruction_audit.json",
    "pitcher_mapping_audit.csv",
    "candidate_topk.csv",
    "seed_pitcher_mapping_audit.csv",
    "seed_team_season_mapping.csv",
    "team_season_mapping.csv",
    "coverage_by_season.csv",
    "threshold_sensitivity.csv",
]
REVIEW_ZIP = OUTPUT_ROOT / "trackman_linkage_review.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_files:
        path = RUN_DIR / name
        if not path.exists():
            raise FileNotFoundError(path)
        archive.write(path, arcname=name)
with zipfile.ZipFile(REVIEW_ZIP) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"review ZIP corruption: {bad_member}")
print("review ZIP:", REVIEW_ZIP)
print("review size (MB):", round(REVIEW_ZIP.stat().st_size / 1e6, 3))